# Vision Transformer (ViT) - TensorFlow / Keras



In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import numpy as np

# -----------------------------
# 1. Load and Preprocess Data
# -----------------------------
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

IMG_SIZE = 32
PATCH_SIZE = 4
NUM_PATCHES = (IMG_SIZE // PATCH_SIZE) ** 2  # 64
EMBED_DIM = 256
NUM_HEADS = 8
MLP_DIM = EMBED_DIM * 4
NUM_LAYERS = 6
NUM_CLASSES = 10
DROPOUT = 0.1

# -----------------------------
# 2. Patch Extraction Layer
# -----------------------------
class PatchExtract(layers.Layer):
    """Extract non-overlapping patches from an image."""

    def __init__(self, patch_size):
        super(PatchExtract, self).__init__()
        self.patch_size = patch_size

    def call(self, images):
        batch_size = tf.shape(images)[0]
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding='VALID')
        patch_dim = patches.shape[-1]
        patches = tf.reshape(patches, [batch_size, -1, patch_dim])
        return patches


# -----------------------------
# 3. Patch Embedding Layer
# -----------------------------
class PatchEmbedding(layers.Layer):
    """Project patches to embedding dim and add positional embeddings."""

    def __init__(self, num_patches, embed_dim):
        super(PatchEmbedding, self).__init__()
        self.num_patches = num_patches
        self.projection = layers.Dense(embed_dim)
        self.cls_token = self.add_weight(
            shape=(1, 1, embed_dim),
            initializer='zeros',
            trainable=True,
            name='cls_token')
        self.position_embedding = self.add_weight(
            shape=(1, num_patches + 1, embed_dim),
            initializer='zeros',
            trainable=True,
            name='position_embedding')

    def call(self, patches):
        batch_size = tf.shape(patches)[0]

        # Project patches
        x = self.projection(patches)  # (B, N, D)

        # Prepend [CLS] token
        cls_tokens = tf.broadcast_to(
            self.cls_token, [batch_size, 1, tf.shape(x)[-1]])  # (B, 1, D)
        x = tf.concat([cls_tokens, x], axis=1)  # (B, N+1, D)

        # Add positional embeddings
        x = x + self.position_embedding
        return x


# -----------------------------
# 4. Transformer Encoder Block
# -----------------------------
class TransformerBlock(layers.Layer):
    """Pre-LN Transformer: LN -> MHSA -> Residual -> LN -> MLP -> Residual."""

    def __init__(self, embed_dim, num_heads, mlp_dim, dropout=0.1):
        super(TransformerBlock, self).__init__()

        self.ln1 = layers.LayerNormalization(epsilon=1e-6)
        self.mhsa = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim // num_heads, dropout=dropout)

        self.ln2 = layers.LayerNormalization(epsilon=1e-6)
        self.mlp = tf.keras.Sequential([
            layers.Dense(mlp_dim, activation='gelu'),
            layers.Dropout(dropout),
            layers.Dense(embed_dim),
            layers.Dropout(dropout),
        ])

    def call(self, x, training=False):
        # Multi-Head Self-Attention with residual
        x_norm = self.ln1(x)
        attn_out = self.mhsa(x_norm, x_norm, training=training)
        x = x + attn_out

        # Feed-Forward MLP with residual
        x = x + self.mlp(self.ln2(x), training=training)
        return x


# -----------------------------
# 5. Build ViT Model
# -----------------------------
def build_vit(img_size=32, patch_size=4, num_classes=10,
              embed_dim=256, depth=6, num_heads=8,
              mlp_dim=1024, dropout=0.1):

    num_patches = (img_size // patch_size) ** 2

    inputs = layers.Input(shape=(img_size, img_size, 3))

    # Extract and embed patches
    patches = PatchExtract(patch_size)(inputs)
    x = PatchEmbedding(num_patches, embed_dim)(patches)
    x = layers.Dropout(dropout)(x)

    # Transformer encoder stack
    for _ in range(depth):
        x = TransformerBlock(embed_dim, num_heads, mlp_dim, dropout)(x)

    # Classification head: take [CLS] token
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    cls_output = x[:, 0]  # (B, D)

    outputs = layers.Dense(num_classes, activation='softmax')(cls_output)

    model = models.Model(inputs, outputs)
    return model


model = build_vit(
    img_size=IMG_SIZE, patch_size=PATCH_SIZE, num_classes=NUM_CLASSES,
    embed_dim=EMBED_DIM, depth=NUM_LAYERS, num_heads=NUM_HEADS,
    mlp_dim=MLP_DIM, dropout=DROPOUT)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'])

model.summary()

# -----------------------------
# 6. Train Model
# -----------------------------
history = model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.1)

# -----------------------------
# 7. Evaluate
# -----------------------------
test_loss, test_acc = model.evaluate(x_test, y_test)
print("Test Accuracy:", test_acc)

# -----------------------------
# 8. Visualization
# -----------------------------
plt.figure(figsize=(12, 5))

# Accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title('Accuracy')
plt.legend()

# Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss')
plt.legend()

plt.show()